In [1]:
# 03_transactions_features
#
# 목적: transactions.csv(2015~2017.2.28 전체 히스토리)를 유저(msno) 단위로 집계해
#      결제 이력 기반 피처를 만든다. 컷오프는 2017-02-28로 통일 (plan 문서 참고).
#      여기서는 결정적 규칙 기반 정제(중복 제거, 이상치 마킹)와 유저별 집계만 수행한다.
#      -> 다른 유저/전체 분포 통계에 의존하지 않으므로 분할 전에 만들어도 리키지 없음.
#
# transactions.csv 컬럼 설명
# msno                    : 유저 ID (1유저 다건 트랜잭션)
# payment_method_id       : 결제 수단 코드
# payment_plan_days       : 결제 주기(일)
# plan_list_price         : 정가(NTD)
# actual_amount_paid      : 실결제액(NTD)
# is_auto_renew           : 자동 갱신 여부
# transaction_date        : 결제 발생일
# membership_expire_date  : 갱신된 멤버십 만료일
# is_cancel               : 능동적 구독 취소 여부 (is_churn과 다른 개념)
#
# 입력: data/raw/transactions.csv
# 출력: data/processed/features_transactions.csv (msno 단위 집계 피처)

In [2]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CUTOFF_DATE = pd.Timestamp("2017-02-28")
MIN_VALID_EXPIRE_DATE = pd.Timestamp("2010-01-01")

In [3]:
df = pd.read_csv(RAW_DIR / "transactions.csv")
print(f"로드: {len(df):,} 행")

before = len(df)
df = df.drop_duplicates()
print(f"완전 중복 제거: {before - len(df):,}건 삭제 -> {len(df):,} 행")

df["transaction_date"] = pd.to_datetime(df["transaction_date"], format="%Y%m%d")
df["membership_expire_date"] = pd.to_datetime(df["membership_expire_date"], format="%Y%m%d")

# membership_expire_date 이상치(2010년 이전, 예: 1970-01-01 센티넬) -> 결측 처리
invalid_expire = df["membership_expire_date"] < MIN_VALID_EXPIRE_DATE
print(f"membership_expire_date 이상치: {invalid_expire.sum():,}건 -> 결측 처리")
df.loc[invalid_expire, "membership_expire_date"] = pd.NaT

df["discount"] = df["plan_list_price"] - df["actual_amount_paid"]

로드: 21,547,746 행


완전 중복 제거: 3,339건 삭제 -> 21,544,407 행


membership_expire_date 이상치: 1,783건 -> 결측 처리


In [4]:
# 유저별 전체 히스토리 집계
agg_basic = df.groupby("msno").agg(
    txn_count=("msno", "size"),
    payment_method_nunique=("payment_method_id", "nunique"),
    plan_days_nunique=("payment_plan_days", "nunique"),
    auto_renew_rate=("is_auto_renew", "mean"),
    cancel_count=("is_cancel", "sum"),
    cancel_rate=("is_cancel", "mean"),
    total_amount_paid=("actual_amount_paid", "sum"),
    avg_discount=("discount", "mean"),
)
agg_basic.head()

,txn_count,payment_method_nunique,plan_days_nunique,auto_renew_rate,cancel_count,cancel_rate,total_amount_paid,avg_discount
msno,,,,,,,,
+++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY=,1,1,1,0.0,0,0.0,0,0.000000
+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,1,1,1,0.0,0,0.0,1788,0.000000
+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,4,1,1,1.0,0,0.0,396,0.000000
+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,19,1,3,1.0,0,0.0,2831,-7.842105
+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,26,1,2,1.0,0,0.0,3874,-5.730769


In [5]:
# 가장 최근 트랜잭션 기준 스냅샷 피처
# 동일 유저가 같은 날 같은 만료일로 여러 건 거래한 극소수 잔여 동점까지 완전히 없애기 위해
# actual_amount_paid를 3번째 동점 기준으로 추가한다 (transaction_date -> membership_expire_date
# -> actual_amount_paid 순). 이 규칙은 SQL(analytics/01_sql_feature_mart.ipynb)과 동일하게 맞춘다.
last_txn = (
    df.sort_values(["transaction_date", "membership_expire_date", "actual_amount_paid"])
    .groupby("msno")
    .tail(1)
    .set_index("msno")[
        ["payment_plan_days", "plan_list_price", "actual_amount_paid", "is_auto_renew",
         "transaction_date", "membership_expire_date"]
    ]
    .rename(columns=lambda c: f"last_{c}")
)

last_txn["days_since_last_txn"] = (CUTOFF_DATE - last_txn["last_transaction_date"]).dt.days
last_txn["days_to_expire"] = (last_txn["last_membership_expire_date"] - CUTOFF_DATE).dt.days

last_txn = last_txn.drop(columns=["last_transaction_date", "last_membership_expire_date"])
last_txn.head()

,last_payment_plan_days,last_plan_list_price,last_actual_amount_paid,last_is_auto_renew,days_since_last_txn,days_to_expire
msno,,,,,,
eaDoXXwgzxC8kD1rhOPgHcpOQFZdS1RrjqGUb/jP7uQ=,30,149,0,1,789,-790.0
+ImL9HROChlux3yqeUiV3aFhwg4CpdBaQyZEQgi6NdI=,30,149,0,1,789,-790.0
X4FLsg1ml9tgrSpaTKexJ8eAFC9GflSuRYQIbr4XSXc=,30,149,0,1,789,-790.0
PR4b7q1J1OR0m3UsXleLsBhxhiXiUzxFJqBiQaZ4JLA=,30,149,0,1,789,-790.0
qhi0846UYdfFd8g4+2sFdYpEc6d10f9RzMouMki1Qt4=,30,149,0,1,789,-790.0


In [6]:
features_transactions = agg_basic.join(last_txn, how="left").reset_index()

print(features_transactions.shape)
print(features_transactions.isna().sum())
features_transactions.head()

(2363626, 15)


msno                          0
txn_count                     0
payment_method_nunique        0
plan_days_nunique             0
auto_renew_rate               0
cancel_count                  0
cancel_rate                   0
total_amount_paid             0
avg_discount                  0
last_payment_plan_days        0
last_plan_list_price          0
last_actual_amount_paid       0
last_is_auto_renew            0
days_since_last_txn           0
days_to_expire             1409
dtype: int64


,msno,txn_count,payment_method_nunique,plan_days_nunique,auto_renew_rate,cancel_count,cancel_rate,total_amount_paid,avg_discount,last_payment_plan_days,last_plan_list_price,last_actual_amount_paid,last_is_auto_renew,days_since_last_txn,days_to_expire
0,+++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY=,1,1,1,0.0,0,0.0,0,0.000000,7,0,0,0,172,-167.0
1,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,1,1,1,0.0,0,0.0,1788,0.000000,410,1788,1788,0,465,-55.0
2,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,4,1,1,1.0,0,0.0,396,0.000000,30,99,99,1,13,15.0
3,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,19,1,3,1.0,0,0.0,2831,-7.842105,30,149,149,1,28,19.0
4,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,26,1,2,1.0,0,0.0,3874,-5.730769,30,149,149,1,2,26.0


In [7]:
features_transactions.to_csv(PROCESSED_DIR / "features_transactions.csv", index=False)
print(f"저장 완료: {PROCESSED_DIR / 'features_transactions.csv'} ({len(features_transactions):,} rows, {features_transactions['msno'].nunique():,} unique users)")

저장 완료: ..\data\processed\features_transactions.csv (2,363,626 rows, 2,363,626 unique users)
